# Pricing, availability, and latent demand

When inventory is limited, recorded sales are `min(demand, inventory)`. A price response for sales is therefore not automatically a price response for demand. This notebook makes the observation mechanism and its identifying assumption explicit, estimates the licensed Cox IPCW demand curve, and keeps a true remaining refusal rather than substituting the observed proxy.

The synthetic data are deterministic given the seed. Run from a source checkout after building the Python extension (`cd python && maturin develop --release`). After the 1.3.0 PyPI release, Colab users can `pip install antecedent==1.3.0`.

In [ ]:
import numpy as np

import antecedent
from antecedent import PointDerivative, ResponseCurve, analyze
from antecedent.observation import IndependentGiven, RightCensored

SEED = 505
N = 2_000

## Simulate demand and the sales recording process

Season affects both price and latent demand. Conditional on price and season in this synthetic design, inventory is generated independently of the remaining demand shock. Inventory binds often enough to pull the recorded-sales slope away from demand, without driving fitted censoring survival through the positivity floor. That design detail motivates—but does not prove in real data—the declared independent-censoring assumption below.

In [ ]:
rng = np.random.default_rng(SEED)
season_index = rng.normal(size=N)
price = np.clip(10.0 + 0.9 * season_index + rng.normal(scale=0.8, size=N), 7.0, 13.0)
latent_demand = np.maximum(
    0.0,
    150.0 - 8.0 * price + 18.0 * season_index + rng.normal(scale=8.0, size=N),
)
inventory = np.maximum(
    5.0,
    100.0 - 2.0 * price + 8.0 * season_index + rng.normal(scale=13.0, size=N),
)
observed_sales = np.minimum(latent_demand, inventory)
demand_observed = (latent_demand <= inventory).astype(float)

observed_data = {
    "season_index": season_index,
    "price": price,
    "inventory": inventory,
    "observed_sales": observed_sales,
    "demand_observed": demand_observed,
}
float(1.0 - demand_observed.mean())  # fraction of inventory-censored rows

## The tempting analysis answers the sales question

This query can run through the complete-observation response estimator, but its outcome is explicitly `observed_sales`. Its slope mixes the demand response with inventory constraints. It must not be labelled a demand elasticity or demand derivative.

In [ ]:
sales_graph = [
    ("season_index", "price"),
    ("season_index", "inventory"),
    ("season_index", "observed_sales"),
    ("price", "observed_sales"),
    ("inventory", "observed_sales"),
]
sales_query = PointDerivative("price", "observed_sales", at=10.0)
sales_response = analyze(
    observed_data,
    graph=sales_graph,
    query=sales_query,
    estimator_config={"bandwidth": 0.8},
)

{
    "estimand": "local causal derivative of recorded sales",
    "estimate": sales_response.estimate,
    "support": sales_response.support.status,
    "uncertainty": sales_response.uncertainty.kind,
}

## State the scientific demand query

`RightCensored` describes the recorded columns. `IndependentGiven` is a separate scientific assumption: knowing price and season, the censoring process carries no additional information about latent demand. Merely having `inventory` and `demand_observed` columns would not justify that claim.

In [ ]:
sales_as_censored_demand = RightCensored(
    "latent_demand",
    "observed_sales",
    "inventory",
    "demand_observed",
)
independent_censoring = IndependentGiven(["price", "season_index"])

demand_query = ResponseCurve(
    "price",
    "latent_demand",
    grid=[9.0, 10.0, 11.0],
    observation=sales_as_censored_demand,
    observation_assumptions=[independent_censoring],
)
demand_query

## Licensed Cox IPCW, then a remaining refusal

`IndependentGiven(["price", "season_index"])` is nonempty and contains treatment plus every adjustment variable on this demand graph, so 1.3 composes Cox IPCW into the demand `ResponseCurve`. Cox is a proportional-hazards nuisance on those declared covariates, not a test that independent censoring holds. The result is a point curve: observation correction and curve smoothing do not yield a joint band.

A demand `PointDerivative` under the same observation mechanism is still refused. Observation-adjusted derivatives are not licensed, and Antecedent does not relabel the sales slope as a demand slope.

In [ ]:
demand_graph = [
    ("season_index", "price"),
    ("season_index", "latent_demand"),
    ("price", "latent_demand"),
]
demand = analyze(observed_data, graph=demand_graph, query=demand_query)
{
    "estimand": "price response of latent demand under Cox IPCW",
    "values": demand.response.values,
    "support": demand.support.status,
    "uncertainty": demand.uncertainty.kind,
}

demand_slope = PointDerivative(
    "price",
    "latent_demand",
    at=10.0,
    observation=sales_as_censored_demand,
    observation_assumptions=[independent_censoring],
)
try:
    analyze(observed_data, graph=demand_graph, query=demand_slope)
except ValueError as error:
    message = str(error)
    assert "MeanCurve only" in message
    print("Expected fail-closed result:", message)
else:
    raise AssertionError("observation-adjusted derivative unexpectedly bypassed its guard")

## What can be concluded now

The sales derivative answers recorded sales under complete observation. The demand curve under the declared `IndependentGiven` assumption is the licensed Cox IPCW path; it is a point curve without a joint observation/curve band. A demand derivative under the same observation mechanism is still refused. This notebook does not impute latent demand, treat stocked-out rows as ordinary sales, or claim that the sales slope is a demand slope.